# NimbusPay ticket triage: LoRA fine-tune of Qwen2.5-1.5B-Instruct (VIMA3YA take-home)

**Runs top to bottom on a fresh free-tier Colab T4.** Default path (flags below off): audit + clean data,
build augmentation, retrain the final adapter, evaluate on dev, write `predictions_test.jsonl`, and verify reproducibility.
Expected time on a T4: roughly 50-70 minutes.

Experiments and baselines (Parts B and D) are behind `RUN_BASELINES` / `RUN_EXPERIMENTS`. When off, their tables are
rebuilt from the saved logs in `WORK/` (their code is below and is what produced those logs).

Layout: `src/` holds the code (identical to the repo's `src/`), `WORK/` (Google Drive) holds adapters, logs and predictions
so a Colab disconnect never loses a finished run.

In [21]:
# --- 0. Environment -------------------------------------------------------------------------
!pip -q install -U peft bitsandbytes accelerate && pip -q uninstall -y torchao  # old Colab torchao breaks new peft; peft does not need it
import os, sys, json, time, shutil, random, collections, subprocess
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/2**30:.1f} GiB")

torch 2.14.1+cu130 | transformers 5.22.1 | peft 0.21.2
Tesla T4 14.6 GiB


In [22]:
# --- 1. Paths and switches -----------------------------------------------------------------
USE_DRIVE = False   # True: persist everything on Google Drive (survives disconnects). False: keep in /content
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/vima3ya"
else:
    ROOT = "/content/vima3ya" if os.path.isdir("/content") else os.path.abspath("vima3ya")
PACK = f"{ROOT}/candidate_pack"   # must contain data/{train,dev,test_inputs}.jsonl, score.py, SCHEMA.md
WORK = f"{ROOT}/work"             # adapters / logs / predictions persist here
SUB  = "/content/submission" if os.path.isdir("/content") else os.path.abspath("submission")
for d in (WORK, SUB, "src", "data_proc"): os.makedirs(d, exist_ok=True)
DATA = f"{PACK}/data"
os.makedirs(DATA, exist_ok=True)
NEED = ["train.jsonl", "dev.jsonl", "test_inputs.jsonl", "score.py", "SCHEMA.md"]
dest = lambda f: f"{DATA}/{f}" if f.endswith(".jsonl") else f"{PACK}/{f}"
if not all(os.path.exists(dest(f)) for f in NEED):
    try:
        from google.colab import files
        print("Upload: train.jsonl, dev.jsonl, test_inputs.jsonl, score.py, SCHEMA.md")
        for name, data in files.upload().items():
            open(dest(name), "wb").write(data)
    except ImportError:
        pass
for f in NEED:
    assert os.path.exists(dest(f)), f"missing {dest(f)}"

RUN_BASELINES   = False   # Part B (about 10 min). The saved outputs below come from a run with this True
RUN_EXPERIMENTS = False   # Part D (about 90 min). The saved outputs below come from a run with this True
FINAL = "E5_aug_lr5e-4"   # which experiment config is the submitted model (set from the Part D table)
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
SEED = 42

In [23]:
%%writefile src/speccheck.py
"""Rule-based implementation of SCHEMA.md (v2), used ONLY to audit train labels.

It is never used at inference time (the brief forbids post-processing).
Every rule below is traceable to a line in SCHEMA.md; anything the spec does not
pin down is marked AMBIGUOUS and is decided by the train-label majority, not by dev.
"""
import datetime as dt
import re

MONTHS = {m: i + 1 for i, m in enumerate(
    ["jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"])}

HINDI_WORDS = set("""
hai hain tha thi nahi nahin kar karo karna karun kaise kya kyun mera mere meri mujhe
ho gaya gayi gaye raha rahi rahe hua hui se ka ki ke ko bhi abhi yeh ye aur par lekin
paise paisa rupaye kat kaat liye diya wapas chahiye kab tak baar baat pehle mein
jaldi bhai dost usko mile mila aaya aayi bata batao kahan wala wali dikha karke liya
""".split())

LEGAL = re.compile(r"(?i)\b(rbi|ombudsman|consumer\s+court|lawyer|legal\s+notice|legal\s+action|vakil|court)\b")
REPEAT = re.compile(
    r"(?i)(third time|second time|again and again|baar baar|teesri baar|dusri baar|doosri baar|"
    r"contacted (?:support|you) (?:about this )?before|second complaint|already written|"
    r"pehle bhi complaint|already (?:raised|complained|contacted)|kai baar|phir se complaint|"
    r"\d+(?:st|nd|rd|th)? time)")


def split_ticket(user):
    head, body = user.split("\n\n", 1)
    m = re.match(r"Received: (\d{4}-\d\d-\d\d) \(\w{3}\)\nSource: (\S+)", head)
    recv = dt.date.fromisoformat(m[1])
    lines = body.split("\n")
    own = "\n".join(l for l in lines if not l.lstrip().startswith(">"))
    return recv, m[2], own


# ---------------- txn_id ----------------
TXN = re.compile(r"(?i)(?<![a-z])n[\s\-_.#:]*p[\s\-_.#:]*((?:\d[\s\-]*){10})(?!\d)")


def txn_ids(text):
    out = []
    for m in TXN.finditer(text):
        d = re.sub(r"\D", "", m[1])
        if len(d) == 10:
            out.append("NP" + d)
    return out


# ---------------- amount ----------------
NUM = r"(\d[\d,]*(?:\.\d+)?)"
MULT = r"(k|lakhs?)\b"
AMT = re.compile(
    r"(?i)(?:(?:\brs\.?|\binr|₹)\s*" + NUM + r"(?:\s*/-)?(?:\s*" + MULT + r")?)"
    r"|(?:(?<![\d,.])" + NUM + r"\s*" + MULT + r")"
    r"|(?:(?<![\d,.])" + NUM + r"\s*(?:rupees|rupaye|rupay|rs)\b)")
DISTRACT = re.compile(r"(?i)(balance|limit|fee\b|fees|pade hain|bacha|minimum|maximum)")


def _val(num, mult):
    x = float(num.replace(",", ""))
    if mult and mult.lower() == "k":
        x *= 1000
    elif mult and mult.lower().startswith("lakh"):
        x *= 100000
    return int(round(x * 100))


def amounts(text):
    """Return (kept, distractor) amounts in paise.

    An amount is a distractor if a balance/limit/fee word appears between the previous
    amount (or sentence start) and it, or in the 15 characters right after it.
    NP transaction ids are masked first so their digits are never read as money.
    """
    masked = TXN.sub(lambda m: "#" * len(m[0]), text)
    keep, drop = [], []
    sents = re.split(r"(?<![Rr]s\.)(?<=[.!?])\s+|\n", masked)
    for s in sents:
        prev = 0
        for m in AMT.finditer(s):
            if m[1]:
                v = _val(m[1], m[2])
            elif m[3]:
                v = _val(m[3], m[4])
            else:
                v = _val(m[5], None)
            ctx = s[prev:m.start()] + " " + s[m.end():m.end() + 15]
            (drop if DISTRACT.search(ctx) else keep).append(v)
            prev = m.end()
    return keep, drop


# ---------------- txn_date ----------------
def _most_recent(day, month, recv):
    for y in (recv.year, recv.year - 1):
        try:
            d = dt.date(y, month, day)
        except ValueError:
            continue
        if d <= recv:
            return d
    return None


DATE_PATTERNS = [
    (re.compile(r"\b(\d{4})-(\d\d)-(\d\d)\b"), "iso"),
    (re.compile(r"\b(\d{1,2})[-/](\d{1,2})[-/](\d{4})\b"), "dmy4"),
    (re.compile(r"\b(\d{1,2})[-/](\d{1,2})[-/](\d{2})\b"), "dmy2"),
    (re.compile(r"(?i)\b(\d{1,2})(?:st|nd|rd|th)?\s+(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?(?:,?\s+(20\d\d))?\b"), "dMy"),
    (re.compile(r"(?i)\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\.?\s+(\d{1,2})(?:st|nd|rd|th)?(?:,?\s+(20\d\d))?\b"), "Mdy"),
    (re.compile(r"(?i)\bday before yesterday\b|\bparso\b"), "rel2"),
    (re.compile(r"(?i)\byesterday\b|\bkal\b"), "rel1"),
    (re.compile(r"(?i)\btoday\b|\baaj\b"), "rel0"),
    (re.compile(r"(?i)\b(\d+)\s+(?:days?\s+ago|din\s+pehle)\b"), "relN"),
    (re.compile(r"(?i)\b(?:a|one|ek)\s+(?:week\s+ago|hafte\s+pehle|hafta\s+pehle)\b"), "rel7"),
]


def txn_dates(text, recv):
    found = []
    taken = []
    for pat, kind in DATE_PATTERNS:
        for m in pat.finditer(text):
            if any(a < m.end() and m.start() < b for a, b in taken):
                continue
            d = None
            try:
                if kind == "iso":
                    d = dt.date(int(m[1]), int(m[2]), int(m[3]))
                elif kind == "dmy4":
                    d = dt.date(int(m[3]), int(m[2]), int(m[1]))
                elif kind == "dmy2":
                    d = dt.date(2000 + int(m[3]), int(m[2]), int(m[1]))
                elif kind == "dMy":
                    day, mon = int(m[1]), MONTHS[m[2][:3].lower()]
                    d = dt.date(int(m[3]), mon, day) if m[3] else _most_recent(day, mon, recv)
                elif kind == "Mdy":
                    day, mon = int(m[2]), MONTHS[m[1][:3].lower()]
                    d = dt.date(int(m[3]), mon, day) if m[3] else _most_recent(day, mon, recv)
                elif kind == "rel0":
                    d = recv
                elif kind == "rel1":
                    d = recv - dt.timedelta(days=1)
                elif kind == "rel2":
                    d = recv - dt.timedelta(days=2)
                elif kind == "relN":
                    d = recv - dt.timedelta(days=int(m[1]))
                elif kind == "rel7":
                    d = recv - dt.timedelta(days=7)
            except ValueError:
                d = None
            if d:
                taken.append((m.start(), m.end()))
                found.append((m.start(), d.isoformat(), kind))
    found.sort()
    return [(d, k) for _, d, k in found]


# ---------------- channel ----------------
CHAN = [
    ("netbanking", re.compile(r"(?i)\b(net\s?banking|internet banking)\b")),
    ("upi", re.compile(r"(?i)\bupi\b")),
    ("wallet", re.compile(r"(?i)\bwallet\b")),
    # Train labels (235/235 rows) treat ANY mention of "card" as channel=card, except
    # "scratch card" (a reward, not a payment mode). AMBIGUOUS in spec; train majority decides.
    ("card", re.compile(r"(?i)(?<!scratch )(?<!srcatch )\bcards?\b|\bvisa\b|\bmastercard\b")),
]


def channels(text):
    return [c for c, p in CHAN if p.search(text)]


# ---------------- language ----------------
def hinglish_score(text):
    toks = re.findall(r"[a-z]+", text.lower())
    if not toks:
        return 0.0
    return sum(t in HINDI_WORDS for t in toks) / len(toks)


# ---------------- derived fields ----------------
def derive_priority(cat, amount, legal):
    if cat == "fraud" or (amount is not None and amount >= 50_000 * 100) or legal:
        return "P1"
    if (cat in ("payment_failed", "refund") and amount is not None and amount >= 5_000 * 100) or cat == "account_access":
        return "P2"
    return "P3"


def derive_needs_human(priority, repeat):
    return priority == "P1" or bool(repeat)


def check(user):
    """Extract everything the spec lets us compute from text alone (category excluded)."""
    recv, src, own = split_ticket(user)
    keep, drop = amounts(own)
    return {
        "recv": recv, "source": src, "own": own,
        "txn_ids": txn_ids(own),
        "amounts": keep, "distractor_amounts": drop,
        "dates": txn_dates(own, recv),
        "channels": channels(own),
        "legal": bool(LEGAL.search(own)),
        "repeat": bool(REPEAT.search(own)),
        "hinglish": hinglish_score(own),
    }

Overwriting src/speccheck.py


In [24]:
%%writefile src/audit.py
"""Compare every parseable train label with the spec checker, field by field.

usage: python -I audit.py <data_dir> [field_to_dump] [max_examples]
"""
import collections
import json
import sys

import os; sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import speccheck as S  # noqa: E402

D = sys.argv[1]
DUMP = sys.argv[2] if len(sys.argv) > 2 else None
NMAX = int(sys.argv[3]) if len(sys.argv) > 3 else 15
NONTXN = {"account_access", "kyc", "other"}


def load(n):
    return [json.loads(l) for l in open(f"{D}/{n}.jsonl", encoding="utf-8")]


def issues(lab, c):
    """Return {field: (label_value, expected_or_hint)} for disagreements."""
    out = {}
    cat = lab.get("category")
    pri = lab.get("priority", lab.get("prio"))
    # transaction fields must be null for non-transaction categories
    if cat in NONTXN:
        for k in ("amount_paise", "txn_date", "txn_id", "channel"):
            if lab.get(k) is not None:
                out[k + "_nonnull"] = (lab.get(k), None)
    else:
        ids = c["txn_ids"]
        exp = ids[0] if len(ids) == 1 else (None if not ids else ids)
        if lab.get("txn_id") != exp:
            out["txn_id"] = (lab.get("txn_id"), exp)
        am = c["amounts"]
        exp = am[0] if len(am) == 1 else (None if not am else am)
        if lab.get("amount_paise") != exp:
            out["amount_paise"] = (lab.get("amount_paise"), exp)
        ds = [d for d, _ in c["dates"]]
        exp = ds[0] if len(set(ds)) == 1 else (None if not ds else ds)
        if lab.get("txn_date") != exp:
            out["txn_date"] = (lab.get("txn_date"), exp)
        ch = c["channels"]
        exp = ch[0] if len(ch) == 1 else (None if not ch else ch)
        if lab.get("channel") != exp:
            out["channel"] = (lab.get("channel"), exp)
    # priority recomputed from the label's own category + amount + text legal threat
    amt = lab.get("amount_paise") if isinstance(lab.get("amount_paise"), int) else None
    exp_p = S.derive_priority(cat, amt, c["legal"])
    if pri != exp_p:
        out["priority"] = (pri, exp_p)
    exp_h = S.derive_needs_human(exp_p, c["repeat"])
    if lab.get("needs_human") != exp_h:
        out["needs_human"] = (lab.get("needs_human"), exp_h)
    exp_l = "hi-en" if c["hinglish"] >= 0.05 else "en"
    if lab.get("language") != exp_l:
        out["language"] = (lab.get("language"), f"{exp_l} (score {c['hinglish']:.2f})")
    return out


def main():
    rows = load("train")
    agg = collections.Counter()
    dump = []
    for r in rows:
        try:
            lab = json.loads(r["messages"][2]["content"])
        except json.JSONDecodeError:
            agg["unparseable"] += 1
            continue
        c = S.check(r["messages"][1]["content"])
        iss = issues(lab, c)
        for k in iss:
            agg[k] += 1
        if DUMP and any(k.startswith(DUMP) for k in iss):
            dump.append((r["id"], {k: v for k, v in iss.items() if k.startswith(DUMP)}, c["own"]))
    print("disagreements per field:", dict(agg.most_common()))
    for rid, iss, own in dump[:NMAX]:
        print(f"\n## {rid} {iss}\n{own.strip()[:600]}")
    if DUMP:
        print(f"\n[{len(dump)} rows flagged for {DUMP}]")


if __name__ == "__main__":
    main()

Overwriting src/audit.py


In [25]:
%%writefile src/clean.py
"""Clean train.jsonl using only train.jsonl + SCHEMA.md (dev is read ONLY to find and drop leaked rows).

usage: python -I clean.py <data_dir> <out_dir>
writes: <out_dir>/train_clean.jsonl, <out_dir>/cleaning_log.csv, prints a summary.

Order matters and is fixed:
  1 drop junk input   2 drop dev leakage   3 drop near-duplicate copies
  4 repair truncated JSON   5 v1-schema rows -> v2   6 txn_id normalisation
  7 amount in rupees -> paise   8 priority recomputed from rules   9 needs_human recomputed
"""
import collections
import csv
import json
import os
import re
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import speccheck as S  # noqa: E402

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
NONTXN = {"account_access", "kyc", "other"}
V1_PRIO = {"low": "P3", "medium": "P2", "high": "P1"}
JUNK = re.compile(r"^\s*((test|asdf)(\s+(test|asdf|\d+))*)?\s*$", re.I)


def norm_body(user):
    b = user.split("\n\n", 1)[1]
    b = re.sub(r"\s+", " ", b).strip().lower()
    return re.sub(r"(\s*(thx|thanks\.?))+$", "", b).strip()


def dup_key(user):
    return user.split("\n\n", 1)[0] + "||" + norm_body(user)


def language_of(c):
    # en rows score exactly 0.000, hi-en rows score >= 0.095 on train; 0.05 sits in the gap.
    return "hi-en" if c["hinglish"] >= 0.05 else "en"


def single(xs):
    xs = list(dict.fromkeys(xs))
    return xs[0] if len(xs) == 1 else (None if not xs else "AMBIG")


def reconstruct(prefix, c):
    """Rebuild a full label for a truncated row from its surviving prefix + the spec checker.
    Returns (label, None) or (None, reason_for_drop)."""
    got = dict(re.findall(r'"(\w+)": ("[^"]*"|null|true|false|-?\d+)(?=,)', prefix))
    got = {k: json.loads(v) for k, v in got.items()}
    cat = got.get("category") or re.match(r'\{"category": "(\w+)"', prefix)[1]
    lab = {"category": cat}
    if cat in NONTXN:
        amt = date = tid = ch = None
    else:
        amt, tid, ch = single(c["amounts"]), single(c["txn_ids"]), single(c["channels"])
        date = single([d for d, _ in c["dates"]])
        if "AMBIG" in (amt, tid, ch, date):
            return None, "truncated label and ticket is ambiguous for a transaction field"
    legal = c["legal"]
    lab["priority"] = S.derive_priority(cat, amt, legal)
    lab.update(amount_paise=amt, txn_date=date, txn_id=tid, channel=ch, language=language_of(c))
    lab["needs_human"] = S.derive_needs_human(lab["priority"], c["repeat"])
    lab = {k: lab[k] for k in KEYS}
    # every field that survived truncation must agree (after known-noise normalisation)
    for k, v in got.items():
        if k == "priority":
            continue  # priority noise is a known, separately-fixed problem
        if k == "txn_id" and isinstance(v, str):
            v = "NP" + re.sub(r"\D", "", v)
        if k in lab and lab[k] != v:
            return None, f"truncated label; surviving field {k}={v!r} disagrees with spec reconstruction {lab[k]!r}"
    return lab, None


def main():
    D, OUT = sys.argv[1], sys.argv[2]
    os.makedirs(OUT, exist_ok=True)
    load = lambda n: [json.loads(l) for l in open(f"{D}/{n}.jsonl", encoding="utf-8")]
    train = load("train")
    dev_keys = {dup_key(r["messages"][1]["content"]) for r in load("dev")}
    test_keys = {dup_key(r["messages"][1]["content"]) for r in load("test_inputs")}

    log, kept, seen = [], [], {}
    stats = collections.Counter()

    def drop(rid, why, tag):
        log.append((rid, "drop", why))
        stats["drop:" + tag] += 1

    for r in train:
        rid, user, raw = r["id"], r["messages"][1]["content"], r["messages"][2]["content"]
        c = S.check(user)
        # 1 junk
        if JUNK.match(c["own"]):
            drop(rid, f"junk input text {c['own'].strip()!r}; label cannot be grounded in the ticket", "junk")
            continue
        # 2 leakage
        k = dup_key(user)
        if k in dev_keys:
            drop(rid, "ticket duplicates a dev row (same header and body up to whitespace/'Thx'); leakage", "dev_leak")
            continue
        if k in test_keys:
            drop(rid, "ticket duplicates a test row; leakage", "test_leak")
            continue
        # 3 near-duplicates within train
        if k in seen:
            drop(rid, f"duplicate of {seen[k]} (same header and body up to whitespace/'Thx'); same label", "duplicate")
            continue
        seen[k] = rid

        fixes = []
        # 4 truncated JSON
        try:
            lab = json.loads(raw)
        except json.JSONDecodeError:
            lab, why = reconstruct(raw, c)
            if lab is None:
                drop(rid, why, "truncated_ambiguous")
                continue
            fixes.append("label JSON truncated; rebuilt from surviving prefix + SCHEMA rules (surviving fields verified)")
            stats["fix:truncated"] += 1
        # 5 v1 schema
        if "prio" in lab:
            lab["priority"] = V1_PRIO[lab.pop("prio")]
            fixes.append("v1 schema: key 'prio' with low/medium/high renamed to 'priority' P3/P2/P1")
            stats["fix:v1_prio"] += 1
            d = lab.get("txn_date")
            if isinstance(d, str) and re.fullmatch(r"\d\d/\d\d/\d{4}", d):
                dd, mm, yy = d.split("/")
                lab["txn_date"] = f"{yy}-{mm}-{dd}"
                fixes.append(f"v1 schema: txn_date {d} reformatted to YYYY-MM-DD")
                stats["fix:v1_date"] += 1
        # 6 txn_id format
        t = lab.get("txn_id")
        if isinstance(t, str) and not re.fullmatch(r"NP\d{10}", t):
            nt = "NP" + re.sub(r"\D", "", t)
            fixes.append(f"txn_id {t!r} normalised to {nt}")
            lab["txn_id"] = nt
            stats["fix:txn_id"] += 1
        # 7 amount unit
        a = lab.get("amount_paise")
        exp = single(c["amounts"])
        if lab["category"] not in NONTXN and isinstance(a, int) and isinstance(exp, int) and a != exp:
            if a == exp // 100:
                fixes.append(f"amount_paise {a} is in rupees; ticket amount is {exp} paise")
                lab["amount_paise"] = exp
                stats["fix:amount_rupees"] += 1
            else:
                drop(rid, f"amount_paise {a} disagrees with ticket amount {exp} for unknown reason", "amount_unknown")
                continue
        # 8 priority
        p = S.derive_priority(lab["category"], lab["amount_paise"], c["legal"])
        if lab["priority"] != p:
            fixes.append(f"priority {lab['priority']} violates SCHEMA rules; set to {p}")
            lab["priority"] = p
            stats["fix:priority"] += 1
        # 9 needs_human
        h = S.derive_needs_human(lab["priority"], c["repeat"])
        if lab["needs_human"] != h:
            fixes.append(f"needs_human {lab['needs_human']} violates SCHEMA rules; set to {h}")
            lab["needs_human"] = h
            stats["fix:needs_human"] += 1

        lab = {k2: lab[k2] for k2 in KEYS}
        if fixes:
            log.append((rid, "fix", "; ".join(fixes)))
        msgs = [r["messages"][0], r["messages"][1], {"role": "assistant", "content": json.dumps(lab)}]
        kept.append({"id": rid, "messages": msgs})

    with open(f"{OUT}/train_clean.jsonl", "w", encoding="utf-8") as f:
        for r in kept:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    with open(f"{OUT}/cleaning_log.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["id", "action", "reason"])
        w.writerows(sorted(log))
    print(f"in {len(train)}  kept {len(kept)}  dropped {sum(1 for x in log if x[1] == 'drop')}  "
          f"fixed rows {sum(1 for x in log if x[1] == 'fix')}")
    for k2, v in sorted(stats.items()):
        print(f"  {k2:28} {v}")


if __name__ == "__main__":
    main()

Overwriting src/clean.py


In [26]:
%%writefile src/aug.py
"""Spec-driven augmentation (Part D, experiment 4). Uses ONLY train_clean.jsonl + SCHEMA.md.

Why: several rules in SCHEMA.md are rare or absent in train (parso 5 rows, kal 13, fee distractors 9,
lawyer/legal-notice 12, Visa/Mastercard 4, order numbers 0, year-less dates that roll back to the
previous year 0, quoted threads 36). We create variants of clean train tickets that exercise exactly
those rules, and recompute every non-category field with speccheck.py (which agrees with 100% of
dev labels on those fields). Category is copied from the seed; no transform changes the intent.

usage: python -I aug.py <train_clean.jsonl> <dev.jsonl> <test_inputs.jsonl> <out.jsonl> [n] [seed]
"""
import datetime as dt
import json
import random
import re
import sys
import os

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import speccheck as S  # noqa: E402

KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
NONTXN = {"account_access", "kyc", "other"}
MON = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
MONTH_FULL = ["January", "February", "March", "April", "May", "June", "July", "August",
              "September", "October", "November", "December"]

# ---- sentence banks (English / Hinglish). Most are harvested from train; the few marked SPEC are
# written from SCHEMA.md wording because train has no example of that rule.
LEGAL = {
    "en": ["I will complain to the RBI ombudsman.", "I will escalate this to the RBI.", "I will go to consumer court.",
           "My lawyer will send a legal notice.", "Otherwise I will approach the consumer court.",
           "I will take legal action.",  # SPEC
           "I will file a complaint with the banking ombudsman.",  # SPEC
           "You will receive a legal notice from my lawyer."],  # SPEC
    "hi-en": ["Consumer court jaunga.", "RBI mein complaint kar dunga.", "Lawyer se legal notice bhejunga.",  # SPEC
              "Ombudsman ke paas complaint karunga.", "Legal action lunga."],  # SPEC
}
REPEAT = {
    "en": ["This is the third time I am contacting you.", "I keep raising this again and again.",
           "This is my second complaint on the same issue.", "I have contacted support about this before.",
           "I have already written {n} times about this."],
    "hi-en": ["Teesri baar bol raha hoon.", "Baar baar complaint kar raha hoon.", "Pehle bhi complaint kar chuka hoon."],
}
DISTRACT = {
    "en": ["My account balance is {a} right now.", "FYI my daily limit is {a}.", "There was a fee of {a} on it.",  # fee: SPEC
           "The convenience fee was {a}.", "My monthly limit is {a}."],
    "hi-en": ["Account mein abhi {a} pade hain.", "Meri daily limit {a} hai.", "Fee {a} lagi thi."],
}
REFNUM = {
    "en": ["Bank ref no {b}.", "My earlier ticket number is #{t}.", "My order number is #{o}.",  # order: SPEC
           "Order ID {o}.", "UTR {b}."],
    "hi-en": ["Bank ref {b} hai.", "Pichla ticket number #{t} tha.", "Order number #{o} hai."],
}
CARDNET = {"en": ["I paid with my Visa card.", "It was a Mastercard payment."],
           "hi-en": ["Visa card se payment kiya tha.", "Mastercard se hua tha."]}
DATE_ABS = {"en": ["This happened on {d}.", "Transaction date {d}.", "Dated {d}.", "It took place on {d}.",
                   "Date of transaction: {d}."],
            "hi-en": ["Yeh {d} ko hua.", "{d} ki transaction hai.", "Transaction date {d} hai."]}
DATE_REL = {"en": {0: ["It was today.", "This happened today."], 1: ["It took place yesterday.", "That was yesterday."],
                   2: ["This happened day before yesterday.", "That was day before yesterday."],
                   "n": ["This happened {n} days ago.", "It was {n} days ago."]},
            "hi-en": {0: ["Yeh aaj hua."], 1: ["kal hi hua yeh.", "kal ki baat hai.", "Yeh kal hua."],
                      2: ["Yeh parso hua.", "parso ki baat hai."], "n": ["{n} din pehle ki baat hai.", "Yeh {n} din pehle hua."]}}
QUOTE = ("> On {d}, NimbusPay Support wrote:\n> Thank you for contacting NimbusPay. Your request has been registered "
         "with ticket #{t}. Our team will get back to you within 48 working hours. Please do not share your OTP, PIN or "
         "card details with anyone, including people who claim to be from NimbusPay. You can track the status of your "
         "request from the Help section of the app. If you have already received a resolution, please ignore this message. "
         "We value your trust and thank you for your patience while we look into this for you.")


def inr(x):  # Indian digit grouping, x in rupees (int)
    s = str(x)
    if len(s) <= 3:
        return s
    head, tail = s[:-3], s[-3:]
    head = re.sub(r"(\d)(?=(\d\d)+$)", r"\1,", head)
    return head + "," + tail


def render_amount(rng, paise):
    """Render an amount in one of the surface forms seen in train (labels are recomputed from text later)."""
    r, p = divmod(paise, 100)
    opts = [lambda: f"Rs {inr(r)}" + (f".{p:02d}" if p else ""), lambda: f"₹{inr(r)}", lambda: f"INR {inr(r)}",
            lambda: f"Rs.{r}/-", lambda: f"{r} rupees", lambda: f"Rs. {r}"]
    if p:  # keep paise exact: only forms that can show decimals
        opts = [lambda: f"Rs {inr(r)}.{p:02d}", lambda: f"₹{inr(r)}.{p:02d}", lambda: f"INR {inr(r)}.{p:02d}"]
    if p == 0 and r % 100 == 0 and r >= 1000:
        opts += [lambda: f"{r / 1000:g}k"] * 3
    if p == 0 and r % 1000 == 0 and r >= 100000:
        opts += [lambda: f"{r / 100000:g} lakh"] * 4
    return rng.choice(opts)()


def pick_amount(rng):
    """Biased to the 5k and 50k priority thresholds and to lakh/k renderings."""
    kind = rng.random()
    if kind < 0.2:
        r = rng.choice([4999, 5000, 5001, 4950, 5100, 49999, 50000, 50001, 49500, 51000])
    elif kind < 0.3:
        r = rng.choice([1, 1.2, 1.5, 2, 2.5, 3.25, 0.5, 4.8]) * 100000
    elif kind < 0.65:
        r = rng.choice(range(1000, 99000, 100))
    else:
        r = round(10 ** rng.uniform(1, 5.5))  # log-uniform 10 .. ~3 lakh rupees, like train
    p = rng.choice([0, 0, 0, 50, 75, 99, 25]) if r < 100000 and rng.random() < 0.3 else 0
    return int(r) * 100 + p


def render_txn(rng, tid):
    d = tid[2:]
    f = rng.choice([lambda: f"np {d[:5]} {d[5:]}", lambda: f"NP-{d}", lambda: f"N P {d[:4]} {d[4:7]} {d[7:]}",
                    lambda: f"np#{d}", lambda: f"NP {d[:5]}-{d[5:]}", lambda: f"Np.{d}", lambda: f"np{d}"])
    return f()


def render_date(rng, d, recv, lang):
    """Render date d (<= recv). Returns sentence or None."""
    delta = (recv - d).days
    if delta in (0, 1, 2) and rng.random() < 0.6:
        return rng.choice(DATE_REL[lang][delta])
    if 3 <= delta <= 9 and rng.random() < 0.5:
        return rng.choice(DATE_REL[lang]["n"]).format(n=delta)
    yearless = rng.random() < 0.55 or d.year < recv.year
    if yearless:
        txt = rng.choice([f"{d.day} {MON[d.month - 1]}", f"{MON[d.month - 1]} {d.day}"])
        if S._most_recent(d.day, d.month, recv) != d:
            return None
    else:
        txt = rng.choice([f"{d.day:02d}-{d.month:02d}-{d.year % 100:02d}", f"{d.day:02d}/{d.month:02d}/{d.year}",
                          f"{d.day}{'th' if 10 <= d.day % 100 <= 20 else {1: 'st', 2: 'nd', 3: 'rd'}.get(d.day % 10, 'th')} {MONTH_FULL[d.month - 1]} {d.year}"])
    return rng.choice(DATE_ABS[lang]).format(d=txt)


def label_for(cat, user):
    """Full label from text using the spec checker; None if any field is ambiguous."""
    c = S.check(user)
    lab = {"category": cat}
    if cat in NONTXN:
        amt = date = tid = ch = None
    else:
        def one(xs):
            xs = list(dict.fromkeys(xs))
            return xs[0] if len(xs) == 1 else (None if not xs else "AMBIG")
        amt, tid, ch = one(c["amounts"]), one(c["txn_ids"]), one(c["channels"])
        date = one([d for d, _ in c["dates"]])
        if "AMBIG" in (amt, tid, ch, date):
            return None
    lab["priority"] = S.derive_priority(cat, amt, c["legal"])
    lab.update(amount_paise=amt, txn_date=date, txn_id=tid, channel=ch,
               language="hi-en" if c["hinglish"] >= 0.05 else "en")
    lab["needs_human"] = S.derive_needs_human(lab["priority"], c["repeat"])
    return {k: lab[k] for k in KEYS}


SENT_SPLIT = re.compile(r"(?<![Rr]s\.)(?<=[.!?])\s+")


def edit_body(user, fn):
    """Apply fn(list_of_sentences)->list to the main body line of the ticket (not quoted lines)."""
    head, body = user.split("\n\n", 1)
    lines = body.split("\n")
    idx = max((i for i, l in enumerate(lines) if not l.startswith(">")), key=lambda i: len(lines[i]))
    sents = [s for s in SENT_SPLIT.split(lines[idx]) if s]
    lines[idx] = " ".join(fn(sents))
    return head + "\n\n" + "\n".join(lines)


def find_sent(sents, pred):
    return [i for i, s in enumerate(sents) if pred(s)]


def transform(rng, row, lang, recv):
    user = row["messages"][1]["content"]
    cat = json.loads(row["messages"][2]["content"])["category"]
    applied = []
    txn = cat not in NONTXN
    # weighted, without replacement; legal/repeat kept rare so the P1/needs_human mix stays close to train
    w = {"legal": 0.6, "repeat": 0.6, "refnum": 2, "quote": 1.5, "distract": 2 if txn else 0.7}
    if txn:
        w.update(amount=3, date=3, txnid=2, card=0.8)
    k = rng.choice([1, 2, 2, 3])
    chosen = []
    while len(chosen) < k and w:
        op = rng.choices(list(w), weights=list(w.values()))[0]
        chosen.append(op)
        w.pop(op)
    for op in chosen:
        if op == "amount":
            def f(s):
                hit = find_sent(s, lambda x: S.amounts(x)[0])
                new = render_amount(rng, pick_amount(rng))
                if hit:
                    i = hit[0]
                    masked = S.TXN.sub(lambda m: "#" * len(m[0]), s[i])
                    m = S.AMT.search(masked)
                    if m:
                        s[i] = s[i][:m.start()] + new + s[i][m.end():]
                        return s
                s.insert(rng.randrange(1, len(s) + 1), rng.choice(["Amount: {a}.", "It was for {a}.", "Rakam {a} hai."
                                                                    if lang == "hi-en" else "{a} is the amount."]).format(a=new))
                return s
            user = edit_body(user, f)
        elif op == "date":
            # biased toward relative dates and year-less dates that resolve to the previous year
            if rng.random() < 0.35:
                # a calendar day later in the year than Received -> a year-less mention means LAST year
                later = [dt.date(recv.year - 1, m, dd) for m in range(recv.month, 13) for dd in range(1, 29)
                         if (m, dd) > (recv.month, recv.day)]
                d = rng.choice(later)
            else:
                d = recv - dt.timedelta(days=rng.choice([0, 1, 1, 2, 2, 3, 4, 5, 6, 8, 12, 20, 35, 60]))
            sent = render_date(rng, d, recv, lang)
            if not sent:
                continue

            def f(s, sent=sent):
                hit = find_sent(s, lambda x: S.txn_dates(x, recv))
                if hit:
                    s[hit[0]] = sent
                else:
                    s.insert(rng.randrange(1, len(s) + 1), sent)
                return s
            user = edit_body(user, f)
        elif op == "txnid":
            tid = "NP" + "".join(rng.choice("0123456789") for _ in range(10))
            txt = render_txn(rng, tid)

            def f(s, txt=txt):
                hit = find_sent(s, lambda x: S.txn_ids(x))
                if hit:
                    s[hit[0]] = S.TXN.sub(txt, s[hit[0]], count=1)
                else:
                    s.insert(rng.randrange(1, len(s) + 1), rng.choice(["Transaction ID {t}.", "Ref: {t}", "{t} check kar lo."
                                                                        if lang == "hi-en" else "The txn id is {t}."]).format(t=txt))
                return s
            user = edit_body(user, f)
        elif op == "card":
            def f(s):
                s = [x for x in s if not S.channels(x)]
                s.insert(rng.randrange(1, len(s) + 1), rng.choice(CARDNET[lang]))
                return s
            user = edit_body(user, f)
        elif op == "legal":
            user = edit_body(user, lambda s: s[:1] + [rng.choice(LEGAL[lang])] + s[1:] if rng.random() < 0.5
                             else s + [rng.choice(LEGAL[lang])])
        elif op == "repeat":
            user = edit_body(user, lambda s: s + [rng.choice(REPEAT[lang]).format(n=rng.randint(2, 5))])
        elif op == "distract":
            kind = rng.choice(["balance", "limit", "fee"] if txn else ["balance", "limit"])
            amt = {"balance": rng.randrange(1000, 20000000, 1), "limit": rng.randrange(5, 300) * 100000,
                   "fee": rng.randrange(500, 50000, 25)}[kind]
            pool = [t for t in DISTRACT[lang] if kind in t.lower() or (kind == "balance" and "pade" in t)]
            sent = rng.choice(pool).format(a=render_amount(rng, amt))
            user = edit_body(user, lambda s, sent=sent: (lambda i: s[:i] + [sent] + s[i:])(rng.randrange(1, len(s) + 1)))
        elif op == "refnum":
            sent = rng.choice(REFNUM[lang]).format(b=rng.randrange(10 ** 11, 10 ** 12), t=rng.randrange(10000, 99999),
                                                   o=rng.randrange(10 ** 7, 10 ** 9))
            user = edit_body(user, lambda s: (lambda i: s[:i] + [sent] + s[i:])(rng.randrange(1, len(s) + 1)))
        elif op == "quote":
            if "Source: email" not in user or "\n>" in user:
                continue
            n = rng.randint(1, 6)
            qs = []
            for _ in range(n):
                qd = recv - dt.timedelta(days=rng.randint(1, 60))
                qs.append(QUOTE.format(d=f"{qd.day:02d} {MON[qd.month - 1]} {qd.year}", t=rng.randint(10000, 99999)))
            user = user.rstrip("\n") + "\n\n" + "\n\n".join(qs)
        applied.append(op)
    return user, applied


def main():
    src, dev_p, test_p, out = sys.argv[1:5]
    n_target = int(sys.argv[5]) if len(sys.argv) > 5 else 700
    rng = random.Random(int(sys.argv[6]) if len(sys.argv) > 6 else 0)
    load = lambda p: [json.loads(l) for l in open(p, encoding="utf-8")]
    train = load(src)
    norm = lambda u: re.sub(r"\s+", " ", u).strip().lower()
    forbidden = {norm(r["messages"][1]["content"]) for r in load(dev_p) + load(test_p)}
    seen = {norm(r["messages"][1]["content"]) for r in train}
    synth, tries, opcount = [], 0, {}
    while len(synth) < n_target and tries < n_target * 20:
        tries += 1
        seed = rng.choice(train)
        lab0 = json.loads(seed["messages"][2]["content"])
        recv = S.split_ticket(seed["messages"][1]["content"])[0]
        user, ops = transform(rng, seed, lab0["language"], recv)
        if not ops or norm(user) in seen or norm(user) in forbidden:
            continue
        lab = label_for(lab0["category"], user)
        if lab is None or lab["language"] != lab0["language"]:
            continue
        seen.add(norm(user))
        for o in ops:
            opcount[o] = opcount.get(o, 0) + 1
        synth.append({"id": f"aug-{len(synth) + 1:05d}", "seed": seed["id"], "ops": ops,
                      "messages": [seed["messages"][0], {"role": "user", "content": user},
                                   {"role": "assistant", "content": json.dumps(lab)}]})
    with open(out, "w", encoding="utf-8") as f:
        for r in train:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
        for r in synth:
            f.write(json.dumps({"id": r["id"], "messages": r["messages"]}, ensure_ascii=False) + "\n")
    with open(out.replace(".jsonl", "_meta.jsonl"), "w", encoding="utf-8") as f:
        for r in synth:
            f.write(json.dumps({"id": r["id"], "seed": r["seed"], "ops": r["ops"]}) + "\n")
    print(f"clean rows {len(train)} + synthetic {len(synth)} (tries {tries}) -> {out}")
    print("ops applied:", dict(sorted(opcount.items())))


if __name__ == "__main__":
    main()

Overwriting src/aug.py


In [27]:
%%writefile src/triage_lib.py
"""Training + inference utilities for the NimbusPay triage take-home.

Plain PyTorch loop on top of HF transformers + PEFT. Written as one small module so every
choice (which tokens get loss, batching, loss normalisation, precision) is visible and testable.
"""
import gc
import json
import math
import os
import random
import subprocess
import time

import torch
import torch.nn.functional as F

BASE_DTYPE = torch.float16  # overridden to float32 only in the CPU unit test

# --------------------------------------------------------------------------------------------
# Data / chat template
# --------------------------------------------------------------------------------------------

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def prompt_ids(tok, row):
    """Exactly what the graders feed the model: system + user, model's own template, gen prompt."""
    msgs = [m for m in row["messages"] if m["role"] in ("system", "user")]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=True, return_dict=False)
    if not isinstance(ids, list):  # transformers 5 can still hand back a BatchEncoding
        ids = ids["input_ids"]
    return list(ids)


def encode_example(tok, row, end_token):
    """input_ids = prompt + label + <end>, labels = -100 on every prompt token (loss on answer only)."""
    p = prompt_ids(tok, row)
    target = row["messages"][-1]["content"] + end_token
    t = tok(target, add_special_tokens=False)["input_ids"]
    return {"input_ids": p + t, "labels": [-100] * len(p) + t, "n_prompt": len(p)}


def check_template(tok, row, end_token):
    """Assert our concatenation equals the model's own template rendering of the full conversation."""
    full = tok.apply_chat_template(row["messages"], tokenize=False)
    ours = tok.apply_chat_template(row["messages"][:2], add_generation_prompt=True, tokenize=False) \
        + row["messages"][-1]["content"] + end_token
    assert full.startswith(ours), f"template mismatch:\n{full!r}\nvs\n{ours!r}"
    tail = full[len(ours):]
    assert tail.strip() == "", f"unexpected text after end token: {tail!r}"
    return full


# --------------------------------------------------------------------------------------------
# Batching: length-bucketed micro-batches under a token budget, grouped into optimizer steps
# --------------------------------------------------------------------------------------------

def make_microbatches(lengths, max_tokens, max_bs, rng):
    """Sort by length (with jitter), cut into micro-batches whose padded size <= max_tokens."""
    order = sorted(range(len(lengths)), key=lambda i: (lengths[i] + rng.random() * 8))
    batches, cur, cur_max = [], [], 0
    for i in order:
        L = lengths[i]
        new_max = max(cur_max, L)
        if cur and (new_max * (len(cur) + 1) > max_tokens or len(cur) >= max_bs):
            batches.append(cur)
            cur, new_max = [], L
        cur.append(i)
        cur_max = new_max
    if cur:
        batches.append(cur)
    rng.shuffle(batches)
    return batches


def group_steps(batches, examples_per_step):
    steps, cur, n = [], [], 0
    for b in batches:
        cur.append(b)
        n += len(b)
        if n >= examples_per_step:
            steps.append(cur)
            cur, n = [], 0
    if cur:
        steps.append(cur)
    return steps


def collate(feats, idx, pad_id, device):
    L = max(len(feats[i]["input_ids"]) for i in idx)
    ids = torch.full((len(idx), L), pad_id, dtype=torch.long)
    lab = torch.full((len(idx), L), -100, dtype=torch.long)
    att = torch.zeros((len(idx), L), dtype=torch.long)
    for r, i in enumerate(idx):  # right padding for training
        x, y = feats[i]["input_ids"], feats[i]["labels"]
        ids[r, :len(x)] = torch.tensor(x)
        lab[r, :len(y)] = torch.tensor(y)
        att[r, :len(x)] = 1
    return ids.to(device), att.to(device), lab.to(device)


# --------------------------------------------------------------------------------------------
# Loss: logits only at supervised positions (saves ~GBs with Qwen's 152k vocab on long tickets)
# --------------------------------------------------------------------------------------------

def supervised_loss_sum(model, ids, att, lab):
    """Sum of token cross-entropy over answer tokens. Next-token shift: hidden[t] predicts token[t+1]."""
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    target = lab[:, 1:]
    mask = target != -100
    try:
        hidden = base.model(input_ids=ids, attention_mask=att, use_cache=False).last_hidden_state
        h = hidden[:, :-1][mask]
        logits = base.lm_head(h).float()
    except AttributeError:  # architecture without .model/.lm_head: fall back to full logits
        logits = model(input_ids=ids, attention_mask=att, use_cache=False).logits[:, :-1][mask].float()
    return F.cross_entropy(logits, target[mask], reduction="sum"), int(mask.sum())


# --------------------------------------------------------------------------------------------
# Model loading
# --------------------------------------------------------------------------------------------

def load_base(model_id, load_4bit, device_map="auto"):
    from transformers import AutoModelForCausalLM, BitsAndBytesConfig
    # T4 has no bf16 -> fp16 weights + fp16 autocast. transformers >= 4.56 renamed `torch_dtype` to `dtype`.
    major, minor = (int(x) for x in __import__("transformers").__version__.split(".")[:2])
    kw = {("dtype" if (major, minor) >= (4, 56) else "torch_dtype"): BASE_DTYPE}
    if load_4bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16)
    if torch.cuda.is_available():
        kw["device_map"] = device_map
    return AutoModelForCausalLM.from_pretrained(model_id, **kw)


def add_lora(model, r, alpha, dropout, target_modules, load_4bit, grad_ckpt=True):
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    if load_4bit:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=grad_ckpt,
                                                gradient_checkpointing_kwargs={"use_reentrant": False})
    elif grad_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    cfg = LoraConfig(r=r, lora_alpha=alpha, lora_dropout=dropout, target_modules=target_modules,
                     bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, cfg)
    # PEFT keeps adapter weights in fp32 by default (autocast_adapter_dtype=True) -> safe with fp16 AMP.
    for n, p in model.named_parameters():
        if p.requires_grad and p.dtype != torch.float32:
            p.data = p.data.float()
    return model


def count_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# --------------------------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------------------------

def train(cfg, tok, train_rows, out_dir, log=print):
    """cfg keys: model_id, load_4bit, r, alpha, dropout, target_modules, lr, epochs, warmup_frac,
    examples_per_step, max_tokens, max_bs, max_len, seed, end_token."""
    from transformers import get_cosine_schedule_with_warmup
    seed = cfg["seed"]
    random.seed(seed)
    torch.manual_seed(seed)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    if dev == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    feats = [encode_example(tok, r, cfg["end_token"]) for r in train_rows]
    lengths = [len(f["input_ids"]) for f in feats]
    too_long = sum(L > cfg["max_len"] for L in lengths)
    assert too_long == 0, f"{too_long} examples exceed max_len={cfg['max_len']}; would truncate the label"

    model = load_base(cfg["model_id"], cfg["load_4bit"])
    model = add_lora(model, cfg["r"], cfg["alpha"], cfg["dropout"], cfg["target_modules"], cfg["load_4bit"],
                     grad_ckpt=cfg.get("grad_ckpt", True))
    n_train = count_trainable(model)
    assert n_train <= 50_000_000, f"{n_train:,} trainable params > 50M cap"
    log(f"trainable params: {n_train:,}")

    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=0.0, betas=(0.9, 0.999))
    rng = random.Random(seed)
    steps_per_epoch = len(group_steps(make_microbatches(lengths, cfg["max_tokens"], cfg["max_bs"], random.Random(0)),
                                      cfg["examples_per_step"]))
    total = steps_per_epoch * cfg["epochs"]
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(cfg["warmup_frac"] * total)), total)
    use_amp = dev == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    history, step = [], 0
    model.train()
    for ep in range(cfg["epochs"]):
        steps = group_steps(make_microbatches(lengths, cfg["max_tokens"], cfg["max_bs"], rng), cfg["examples_per_step"])
        for mbs in steps:
            n_tok = sum(sum(1 for y in feats[i]["labels"][1:] if y != -100) for b in mbs for i in b)
            tot = 0.0
            for b in mbs:
                ids, att, lab = collate(feats, b, tok.pad_token_id, dev)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
                    s, _ = supervised_loss_sum(model, ids, att, lab)
                loss = s / n_tok  # mean over ALL answer tokens in the optimizer step
                scaler.scale(loss).backward()
                tot += loss.item()
            scaler.unscale_(opt)
            gn = torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True)
            sched.step()
            step += 1
            history.append({"step": step, "epoch": ep + 1, "loss": tot, "lr": sched.get_last_lr()[0],
                            "grad_norm": float(gn)})
            if step % 10 == 0 or step == total:
                log(f"ep {ep + 1} step {step}/{total} loss {tot:.4f} lr {sched.get_last_lr()[0]:.2e} gn {float(gn):.2f}")
            if not math.isfinite(tot):
                raise RuntimeError("non-finite loss")
    minutes = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30 if dev == "cuda" else float("nan")
    os.makedirs(out_dir, exist_ok=True)
    model.save_pretrained(out_dir)  # adapter_config.json + adapter_model.safetensors
    json.dump({"cfg": cfg, "train_minutes": minutes, "peak_gpu_gib": peak, "trainable_params": n_train,
               "n_train_rows": len(train_rows), "history": history},
              open(os.path.join(out_dir, "train_log.json"), "w"), indent=1)
    log(f"done in {minutes:.1f} min, peak GPU {peak:.2f} GiB -> {out_dir}")
    del model, opt
    gc.collect()
    if dev == "cuda":
        torch.cuda.empty_cache()
    return {"train_minutes": minutes, "peak_gpu_gib": peak, "trainable_params": n_train}


# --------------------------------------------------------------------------------------------
# Inference (what the graders run: chat template, plain greedy generate, max_new_tokens=200)
# --------------------------------------------------------------------------------------------

def load_for_inference(model_id, adapter_dir, load_4bit):
    from peft import PeftModel
    base = load_base(model_id, load_4bit)
    m = PeftModel.from_pretrained(base, adapter_dir) if adapter_dir else base
    m.eval()
    return m


def end_ids(tok):
    """Stop tokens: chat end-of-turn (<|im_end|> for Qwen) plus the tokenizer eos, as in the hub generation_config."""
    ids = []
    for t in ("<|im_end|>", tok.eos_token):
        i = tok.convert_tokens_to_ids(t) if t else None
        if i is not None and i != tok.unk_token_id and i not in ids:
            ids.append(i)
    return ids


def greedy_config(tok, max_new_tokens=200):
    from transformers import GenerationConfig
    # Explicit config so nothing from the hub generation_config (sampling params, repetition_penalty)
    # leaks in: pure argmax decoding.
    return GenerationConfig(do_sample=False, num_beams=1, max_new_tokens=max_new_tokens, repetition_penalty=1.0,
                            eos_token_id=end_ids(tok), pad_token_id=tok.pad_token_id)


@torch.no_grad()
def generate(model, tok, rows, batch_size=1, max_new_tokens=200, system_override=None, gen_cfg=None, log=print):
    """Return list of {"id","output"}; batch_size=1 means no padding at all (bit-for-bit reproducible path)."""
    dev = next(model.parameters()).device
    gen_cfg = gen_cfg or greedy_config(tok, max_new_tokens)
    old_side = tok.padding_side
    tok.padding_side = "left"
    out, t0 = [], time.time()
    order = sorted(range(len(rows)), key=lambda i: len(rows[i]["messages"][1]["content"]))
    for s in range(0, len(order), batch_size):
        idx = order[s:s + batch_size]
        prompts = []
        for i in idx:
            r = rows[i]
            if system_override is not None:
                r = {"messages": [{"role": "system", "content": system_override}, r["messages"][1]]}
            prompts.append(prompt_ids(tok, r))
        L = max(map(len, prompts))
        ids = torch.full((len(prompts), L), tok.pad_token_id, dtype=torch.long)
        att = torch.zeros((len(prompts), L), dtype=torch.long)
        for k, p in enumerate(prompts):
            ids[k, L - len(p):] = torch.tensor(p)
            att[k, L - len(p):] = 1
        gen = model.generate(input_ids=ids.to(dev), attention_mask=att.to(dev), generation_config=gen_cfg)
        for k, i in enumerate(idx):
            text = tok.decode(gen[k, L:], skip_special_tokens=True)
            out.append((i, {"id": rows[i]["id"], "output": text.strip()}))
        if (s // batch_size) % 20 == 0:
            log(f"  generated {min(s + batch_size, len(order))}/{len(order)} ({time.time() - t0:.0f}s)")
    tok.padding_side = old_side
    out.sort()
    return [o for _, o in out]


def write_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def score(score_py, gold_path, pred_path):
    res = subprocess.run(["python", score_py, "--gold", gold_path, "--pred", pred_path, "--json"],
                         capture_output=True, text=True, check=True)
    return json.loads(res.stdout)

Overwriting src/triage_lib.py


In [28]:
sys.path.insert(0, "src")
import importlib, speccheck, triage_lib as T
importlib.reload(speccheck); importlib.reload(T)

<module 'triage_lib' from '/content/src/triage_lib.py'>

## Part A - Data audit and cleaning

`src/speccheck.py` implements every SCHEMA.md rule that can be computed from text (all fields except `category`).
`src/audit.py` compares it with each train label. Where the spec is ambiguous (e.g. does a bare "my card" count as
`channel=card`?), the train-label majority decides, never dev. Category noise is checked separately (cell below).
Full write-up: `report.md`.

In [29]:
# Raw train vs spec: disagreements per field (before cleaning)
!python -I src/audit.py "{DATA}"

disagreements per field: {'priority': 135, 'txn_date': 34, 'amount_paise': 32, 'unparseable': 25, 'txn_id': 24, 'language': 5, 'needs_human': 4, 'channel': 3}


In [30]:
# Category: out-of-fold TF-IDF classifier + core-sentence grouping; every disagreement was read by hand
!pip -q install scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.pipeline import make_pipeline
import re, numpy as np
raw = T.load_jsonl(f"{DATA}/train.jsonl")
X, y, ids = [], [], []
for r in raw:
    own = speccheck.split_ticket(r["messages"][1]["content"])[2]
    if len(own.strip()) < 15: continue
    X.append(own); y.append(re.search(r'"category": "(\w+)"', r["messages"][2]["content"])[1]); ids.append(r["id"])
pipe = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
                     LogisticRegression(C=10, max_iter=3000))
P = cross_val_predict(pipe, X, y, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")
cls = sorted(set(y)); pred = np.array(cls)[P.argmax(1)]
print(f"OOF accuracy {np.mean(pred == np.array(y)):.3f}; disagreements to review by hand: {int(np.sum(pred != np.array(y)))}")
for i in np.argsort([P[i, cls.index(y[i])] for i in range(len(y))])[:12]:
    if pred[i] != y[i]: print(f"{ids[i]} label={y[i]:15} oof={pred[i]:15} | {X[i][:110]!r}")

OOF accuracy 0.974; disagreements to review by hand: 37
tr-01286 label=account_access  oof=other           | "[IVR transcript] Hi team, My card's frozen, what can I do? Please look into it."
tr-00418 label=fraud           oof=payment_failed  | 'Subject: Help needed\n\nGood morning,\nThere is a payment in the App that is not mine. Mode: debit card. I will g'
tr-00045 label=fraud           oof=offers          | 'Subject: Help needed\n\n\nI AM MISSING CASH FROM MY BANK ACCOUNT ACCORDING TO MY APP. The amount was INR 11,594. '
tr-01128 label=other           oof=account_access  | 'Subject: Complaint\n\nGood morning,\ncan i create my own pin right away. I am on the iOS app. My earlier ticket n'
tr-00032 label=fraud           oof=payment_failed  | 'Good morning, There is a record of payment I never submitted on the app. It is Rs 40,425.75 in total. Transact'
tr-01333 label=fraud           oof=refund          | 'Subject: Problem with NimbusPay\n\nHello,\nI have a charge I did not make. Transac

In [31]:
# Clean -> data_proc/train_clean.jsonl + cleaning_log.csv ; then re-audit (expect no disagreements)
!python -I src/clean.py "{DATA}" data_proc
os.makedirs("data_proc/chk", exist_ok=True); shutil.copy("data_proc/train_clean.jsonl", "data_proc/chk/train.jsonl")
!python -I src/audit.py data_proc/chk
import pandas as pd
log = pd.read_csv("data_proc/cleaning_log.csv")
print(log.action.value_counts().to_dict())
shutil.copy("data_proc/cleaning_log.csv", f"{SUB}/cleaning_log.csv")
log.head(8)

in 1445  kept 1299  dropped 146  fixed rows 200
  drop:dev_leak                45
  drop:duplicate               91
  drop:junk                    10
  fix:amount_rupees            25
  fix:priority                 60
  fix:truncated                25
  fix:txn_id                   20
  fix:v1_date                  30
  fix:v1_prio                  70
disagreements per field: {}
{'fix': 200, 'drop': 146}


,id,action,reason
0,tr-00008,drop,ticket duplicates a dev row (same header and b...
1,tr-00013,fix,priority P1 violates SCHEMA rules; set to P3
2,tr-00018,fix,priority P1 violates SCHEMA rules; set to P3
3,tr-00020,fix,v1 schema: key 'prio' with low/medium/high ren...
4,tr-00023,fix,priority P1 violates SCHEMA rules; set to P3
5,tr-00034,drop,junk input text 'test test 123'; label cannot ...
6,tr-00045,fix,priority P3 violates SCHEMA rules; set to P1
7,tr-00049,fix,v1 schema: key 'prio' with low/medium/high ren...


In [32]:
# Spec-driven augmentation (experiment E4): only train_clean.jsonl + SCHEMA.md rules are used
!python -I src/aug.py data_proc/train_clean.jsonl "{DATA}/dev.jsonl" "{DATA}/test_inputs.jsonl" data_proc/train_aug.jsonl 700 0
os.makedirs("data_proc/chk_aug", exist_ok=True); shutil.copy("data_proc/train_aug.jsonl", "data_proc/chk_aug/train.jsonl")
!python -I src/audit.py data_proc/chk_aug

clean rows 1299 + synthetic 700 (tries 727) -> data_proc/train_aug.jsonl
ops applied: {'amount': 152, 'card': 46, 'date': 174, 'distract': 199, 'legal': 93, 'quote': 55, 'refnum': 289, 'repeat': 100, 'txnid': 130}
disagreements per field: {}


## Part C - Pipeline: chat template, loss masking, sequence length
* **Template:** the model's own `apply_chat_template` (Qwen2.5 has a native `system` role). Training input is
  `template(system, user, add_generation_prompt=True)` + label + `<|im_end|>`, which is asserted to equal the template's own
  rendering of the full 3-message conversation.
* **Loss:** only on the label tokens and the closing `<|im_end|>` (so the model learns to stop). Prompt tokens get -100,
  and logits are computed only at those positions (saves memory with a 152k vocabulary).
* **Sequence length:** measured below on train, dev **and** test inputs. Truncation would cut the label, so `MAX_LEN` must
  cover the longest ticket. Length-bucketed batching keeps the rare 2-3k-token quoted threads from padding everything else.

In [33]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(MODEL_ID)
END = "<|im_end|>"
assert tok.convert_tokens_to_ids(END) in T.end_ids(tok)
print(repr(T.check_template(tok, T.load_jsonl("data_proc/train_clean.jsonl")[0], END)))

'<|im_start|>system\nYou are NimbusPay\'s ticket triage model. Read the ticket and return the triage JSON.<|im_end|>\n<|im_start|>user\nReceived: 2026-07-25 (Sat)\nSource: app-chat\n\nHi, I have a top up that didn\'t go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply.<|im_end|>\n<|im_start|>assistant\n{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>\n'


In [34]:
clean = T.load_jsonl("data_proc/train_clean.jsonl")
aug   = T.load_jsonl("data_proc/train_aug.jsonl")
dev   = T.load_jsonl(f"{DATA}/dev.jsonl")
test  = T.load_jsonl(f"{DATA}/test_inputs.jsonl")
for r in clean[:200]: T.check_template(tok, r, END)
def lens(rows, with_label=True):
    return np.array([len(T.encode_example(tok, r, END)["input_ids"]) if with_label else len(T.prompt_ids(tok, r))
                     for r in rows])
L = {"train_clean": lens(clean), "train_aug": lens(aug), "dev": lens(dev), "test_prompt": lens(test, False)}
lab_len = np.array([len(tok(r["messages"][-1]["content"] + END, add_special_tokens=False)["input_ids"]) for r in clean])
print(f"{'split':12} {'n':>5} {'p50':>6} {'p90':>6} {'p99':>6} {'max':>6}")
for k, v in L.items(): print(f"{k:12} {len(v):5d} {np.percentile(v,50):6.0f} {np.percentile(v,90):6.0f} {np.percentile(v,99):6.0f} {v.max():6d}")
print(f"label tokens: max {lab_len.max()} (max_new_tokens=200 leaves {200-lab_len.max()} spare)")
MAX_LEN = int(np.ceil((max(v.max() for v in L.values()) + 64) / 256) * 256)
print("MAX_LEN =", MAX_LEN, "| share of train rows > 512 tokens:", f"{np.mean(L['train_clean'] > 512):.1%}")

split            n    p50    p90    p99    max
train_clean   1299    167    213   1817   2214
train_aug     1999    175    222   1935   2235
dev            200    172    212   1516   2180
test_prompt    400    108    143   1827   2089
label tokens: max 84 (max_new_tokens=200 leaves 116 spare)
MAX_LEN = 2304 | share of train rows > 512 tokens: 2.8%


In [35]:
BASE_CFG = dict(model_id=MODEL_ID, load_4bit=False,
                r=16, alpha=32, dropout=0.05,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
                lr=2e-4, epochs=2, warmup_frac=0.05, examples_per_step=16,
                max_tokens=8192, max_bs=16, max_len=MAX_LEN, seed=SEED, end_token=END)
TRAIN_FILES = {"clean": "data_proc/train_clean.jsonl", "aug": "data_proc/train_aug.jsonl", "raw": f"{DATA}/train.jsonl"}
EXPERIMENTS = {   # one change at a time vs E0_ref
    "E0_ref":    dict(data="clean", why="reference: cleaned data, 16-bit LoRA r16, lr 2e-4, 2 epochs"),
    "E1_raw":    dict(data="raw",   why="raw train.jsonl instead of cleaned"),
    "E2_qlora":  dict(data="clean", load_4bit=True, why="4-bit NF4 QLoRA instead of fp16 base"),
    "E3_lr1e-4": dict(data="clean", lr=1e-4, why="learning rate 1e-4 instead of 2e-4"),
    "E3_lr5e-4": dict(data="clean", lr=5e-4, why="learning rate 5e-4 instead of 2e-4"),
    "E4_aug":    dict(data="aug",   why="+700 spec-driven synthetic tickets (own idea)"),
    "E5_aug_lr5e-4": dict(data="aug", lr=5e-4, why="final: E4 data + lr 5e-4 (combines the two wins; not a one-change run)"),
}
def cfg_of(name):
    e = dict(EXPERIMENTS[name]); data = e.pop("data"); e.pop("why")
    return {**BASE_CFG, **e}, TRAIN_FILES[data]

def run_experiment(name, eval_bs=16):
    d = f"{WORK}/exp/{name}"; res_p = f"{d}/result.json"
    if os.path.exists(res_p): return json.load(open(res_p))
    cfg, tf = cfg_of(name)
    if not os.path.exists(f"{d}/adapter_model.safetensors"):
        T.train(cfg, tok, T.load_jsonl(tf), d)
    tl = json.load(open(f"{d}/train_log.json"))
    m = T.load_for_inference(MODEL_ID, d, cfg["load_4bit"])
    preds = T.generate(m, tok, dev, batch_size=eval_bs)
    del m; torch.cuda.empty_cache()
    T.write_jsonl(preds, f"{d}/dev_preds.jsonl")
    s = T.score(f"{PACK}/score.py", f"{DATA}/dev.jsonl", f"{d}/dev_preds.jsonl")["all"]
    res = {"name": name, "change": EXPERIMENTS[name]["why"], "dev_exact": s["exact_match"], "dev_field": s["mean_field_acc"],
           "json_valid": s["json_valid"], "fields": s["fields"], "peak_gpu_gib": tl["peak_gpu_gib"],
           "train_min": tl["train_minutes"], "trainable": tl["trainable_params"], "n_rows": tl["n_train_rows"]}
    json.dump(res, open(res_p, "w"), indent=1)
    return res

In [36]:
# Smoke test (about 2 min): 2 optimizer steps + 2 generations, so a bug never costs a full run
_t = time.time()
T.train({**BASE_CFG, "epochs": 1, "examples_per_step": 4}, tok, clean[:8], "/tmp/smoke")
_m = T.load_for_inference(MODEL_ID, "/tmp/smoke", False)
print(T.generate(_m, tok, dev[:2], batch_size=2)); del _m; torch.cuda.empty_cache()
print(f"smoke ok in {time.time()-_t:.0f}s")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768
ep 1 step 1/1 loss 1.8328 lr 2.00e-04 gn 2.78
done in 0.1 min, peak GPU 7.01 GiB -> /tmp/smoke


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  generated 2/2 (16s)
[{'id': 'dv-00001', 'output': '```json\n{\n  "status": "New",\n  "priority": "High",\n  "category": "Card Activation",\n  "description": "User needs assistance with activating and using a card. Previous ticket #54047 is open.",\n  "resolution": "Pending",\n  "resolution_notes": "User needs to follow the activation instructions provided in the previous ticket."\n}\n```'}, {'id': 'dv-00002', 'output': '```json\n{\n  "priority": "normal",\n  "category": "customer_service",\n  "description": "Customer is asking about ID acceptance options for their Realme 11 phone.",\n  "resolution": "Provide information about ID acceptance options for the Realme 11 phone.",\n  "status": "in_progress"\n}\n```'}]
smoke ok in 30s


## Part B - Baselines (dev, `score.py`)
B1 = base model, fixed prompt. B2 = base model with SCHEMA.md appended to the system prompt (reference only).
We also report a *lenient* parse (first `{...}` block) to separate "doesn't know the rules" from "doesn't follow the output format".

In [37]:
def lenient(preds):
    out = []
    for p in preds:
        m = re.search(r"\{.*\}", p["output"], re.S)
        out.append({"id": p["id"], "output": m[0] if m else p["output"]})
    return out

BASE_DIR = f"{WORK}/baselines"; os.makedirs(BASE_DIR, exist_ok=True)
RULES_PROMPT = dev[0]["messages"][0]["content"] + "\n\n" + open(f"{PACK}/SCHEMA.md", encoding="utf-8").read()
if RUN_BASELINES and not os.path.exists(f"{BASE_DIR}/results.json"):
    m = T.load_for_inference(MODEL_ID, None, False)
    res = {}
    for name, sysp in [("B1_fixed_prompt", None), ("B2_rules_in_prompt", RULES_PROMPT)]:
        preds = T.generate(m, tok, dev, batch_size=8, system_override=sysp)
        T.write_jsonl(preds, f"{BASE_DIR}/{name}.jsonl"); T.write_jsonl(lenient(preds), f"{BASE_DIR}/{name}_lenient.jsonl")
        s  = T.score(f"{PACK}/score.py", f"{DATA}/dev.jsonl", f"{BASE_DIR}/{name}.jsonl")["all"]
        sl = T.score(f"{PACK}/score.py", f"{DATA}/dev.jsonl", f"{BASE_DIR}/{name}_lenient.jsonl")["all"]
        res[name] = {"strict": s, "lenient": sl}
    json.dump(res, open(f"{BASE_DIR}/results.json", "w"), indent=1); del m; torch.cuda.empty_cache()
if os.path.exists(f"{BASE_DIR}/results.json"):
    res = json.load(open(f"{BASE_DIR}/results.json"))
    for k, v in res.items():
        print(f"{k:20} strict: valid {v['strict']['json_valid']:.1%} exact {v['strict']['exact_match']:.1%} field {v['strict']['mean_field_acc']:.1%}"
              f" | lenient: exact {v['lenient']['exact_match']:.1%} field {v['lenient']['mean_field_acc']:.1%}")
        print("   per-field (lenient):", {f: round(a, 3) for f, a in v["lenient"]["fields"].items()})
    print("\nexample B1 output:", T.load_jsonl(f"{BASE_DIR}/B1_fixed_prompt.jsonl")[0]["output"][:300])

B1_fixed_prompt      strict: valid 0.0% exact 0.0% field 0.0% | lenient: exact 0.0% field 0.0%
   per-field (lenient): {'category': 0.0, 'priority': 0.0, 'amount_paise': 0.0, 'txn_date': 0.0, 'txn_id': 0.0, 'channel': 0.0, 'language': 0.0, 'needs_human': 0.0}
B2_rules_in_prompt   strict: valid 0.0% exact 0.0% field 0.0% | lenient: exact 0.5% field 61.5%
   per-field (lenient): {'category': 0.67, 'priority': 0.35, 'amount_paise': 0.865, 'txn_date': 0.71, 'txn_id': 0.635, 'channel': 0.75, 'language': 0.635, 'needs_human': 0.305}

example B1 output: ```json
{
  "status": "New",
  "priority": "High",
  "category": "Card Activation",
  "description": "User needs assistance with activating and using a card. Previous ticket #54047 is open.",
  "resolution": "Pending",
  "resolution_notes": "User needs to follow the activation instructions provided 


## Part D - Controlled experiments (one change at a time vs `E0_ref`)

In [38]:
rows = []
for name in EXPERIMENTS:
    if RUN_EXPERIMENTS or os.path.exists(f"{WORK}/exp/{name}/result.json"):
        print(f"=== {name}"); rows.append(run_experiment(name))
if rows:
    tab = pd.DataFrame([{"run": r["name"], "what changed": r["change"], "dev exact": f"{r['dev_exact']:.1%}",
                         "dev mean field": f"{r['dev_field']:.1%}", "peak GPU GiB": f"{r['peak_gpu_gib']:.2f}",
                         "train min": f"{r['train_min']:.1f}", "train rows": r["n_rows"]} for r in rows])
    md_rows = ["| " + " | ".join(tab.columns) + " |", "|" + "---|" * len(tab.columns)]
    md_rows += ["| " + " | ".join(str(v) for v in row) + " |" for row in tab.values]
    open(f"{SUB}/experiments.md", "w").write("\n".join(md_rows) + "\n")
    display(tab)
    display(pd.DataFrame({r["name"]: r["fields"] for r in rows}).round(3))

=== E0_ref
=== E1_raw
=== E2_qlora
=== E3_lr1e-4
=== E3_lr5e-4
=== E4_aug
=== E5_aug_lr5e-4


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768
ep 1 step 10/238 loss 0.2126 lr 4.55e-04 gn 2.34
ep 1 step 20/238 loss 0.0860 lr 4.98e-04 gn 0.41
ep 1 step 30/238 loss 0.0467 lr 4.91e-04 gn 0.22
ep 1 step 40/238 loss 0.0395 lr 4.80e-04 gn 0.09
ep 1 step 50/238 loss 0.0321 lr 4.64e-04 gn 0.19
ep 1 step 60/238 loss 0.0276 lr 4.45e-04 gn 0.20
ep 1 step 70/238 loss 0.0215 lr 4.21e-04 gn 0.09
ep 1 step 80/238 loss 0.0051 lr 3.94e-04 gn 0.12
ep 1 step 90/238 loss 0.0067 lr 3.65e-04 gn 0.10
ep 1 step 100/238 loss 0.0032 lr 3.33e-04 gn 0.05
ep 1 step 110/238 loss 0.0061 lr 3.00e-04 gn 0.12
ep 1 step 120/238 loss 0.0097 lr 2.66e-04 gn 0.13
ep 2 step 130/238 loss 0.0017 lr 2.31e-04 gn 0.05
ep 2 step 140/238 loss 0.0029 lr 1.97e-04 gn 0.07
ep 2 step 150/238 loss 0.0088 lr 1.64e-04 gn 0.12
ep 2 step 160/238 loss 0.0014 lr 1.32e-04 gn 0.06
ep 2 step 170/238 loss 0.0001 lr 1.03e-04 gn 0.00
ep 2 step 180/238 loss 0.0033 lr 7.63e-05 gn 0.06
ep 2 step 190/238 loss 0.0008 lr 5.32e-05 gn 0.03
ep 2 step 200/238 loss 0.0007 

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  generated 16/200 (5s)


,run,what changed,dev exact,dev mean field,peak GPU GiB,train min,train rows
0,E0_ref,"reference: cleaned data, 16-bit LoRA r16, lr 2...",71.0%,95.7%,5.60,9.2,1299
1,E1_raw,raw train.jsonl instead of cleaned,73.5%,95.4%,5.63,10.2,1445
2,E2_qlora,4-bit NF4 QLoRA instead of fp16 base,62.5%,93.6%,5.09,11.2,1299
3,E3_lr1e-4,learning rate 1e-4 instead of 2e-4,59.5%,92.9%,6.86,9.2,1299
4,E3_lr5e-4,learning rate 5e-4 instead of 2e-4,91.5%,98.3%,5.60,9.2,1299
5,E4_aug,+700 spec-driven synthetic tickets (own idea),90.0%,98.2%,5.64,15.3,1999
6,E5_aug_lr5e-4,final: E4 data + lr 5e-4 (combines the two win...,93.5%,98.9%,8.59,15.3,1999


,E0_ref,E1_raw,E2_qlora,E3_lr1e-4,E3_lr5e-4,E4_aug,E5_aug_lr5e-4
category,0.965,0.980,0.970,0.970,0.970,0.975,0.985
priority,0.795,0.755,0.675,0.655,0.930,0.925,0.970
amount_paise,0.995,1.000,0.995,0.985,1.000,0.995,0.995
txn_date,0.980,0.985,0.975,0.960,0.995,0.995,0.980
txn_id,1.000,1.000,0.995,0.995,1.000,1.000,1.000
channel,0.995,1.000,1.000,1.000,1.000,0.995,1.000
language,1.000,1.000,1.000,1.000,1.000,1.000,1.000
needs_human,0.925,0.915,0.875,0.870,0.970,0.970,0.980


## Final model
`FINAL` names an experiment config. If its adapter already exists in `WORK/exp/` it is reused; otherwise (fresh runtime,
e.g. reviewers) it is retrained from scratch. Predictions are then generated **from the adapter re-loaded from disk**, one ticket
at a time (`batch_size=1`, no padding), with plain greedy decoding and `max_new_tokens=200`, the same procedure as the reproducibility check.

In [39]:
cfg, tf = cfg_of(FINAL)
FINAL_DIR = f"{WORK}/exp/{FINAL}"
if not os.path.exists(f"{FINAL_DIR}/adapter_model.safetensors"):
    T.train(cfg, tok, T.load_jsonl(tf), FINAL_DIR)
os.makedirs(f"{SUB}/adapter", exist_ok=True)
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy(f"{FINAL_DIR}/{f}", f"{SUB}/adapter/{f}")
print({f: f"{os.path.getsize(f'{SUB}/adapter/{f}')/2**20:.1f} MiB" for f in os.listdir(f"{SUB}/adapter")})
print("load_4bit =", cfg["load_4bit"])

{'adapter_config.json': '0.0 MiB', 'adapter_model.safetensors': '70.5 MiB'}
load_4bit = False


In [40]:
model = T.load_for_inference(MODEL_ID, f"{SUB}/adapter", cfg["load_4bit"])
t = time.time()
dev_preds = T.generate(model, tok, dev, batch_size=1)
T.write_jsonl(dev_preds, f"{SUB}/predictions_dev.jsonl")
print(f"dev generation {time.time()-t:.0f}s")
!python "{PACK}/score.py" --gold "{DATA}/dev.jsonl" --pred "{SUB}/predictions_dev.jsonl"

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  generated 1/200 (4s)
  generated 21/200 (85s)
  generated 41/200 (170s)
  generated 61/200 (261s)
  generated 81/200 (356s)
  generated 101/200 (464s)
  generated 121/200 (567s)
  generated 141/200 (677s)
  generated 161/200 (789s)
  generated 181/200 (905s)
dev generation 1015s
group                            n   valid   exact  fields
all                            200  100.0%   93.5%   98.9%
category=account_access         19  100.0%  100.0%  100.0%
category=fraud                  15  100.0%  100.0%  100.0%
category=kyc                    24  100.0%  100.0%  100.0%
category=offers                 18  100.0%   94.4%   99.3%
category=other                  16  100.0%  100.0%  100.0%
category=payment_failed         53  100.0%   86.8%   97.9%
category=refund                 55  100.0%   90.9%   98.2%

per-field accuracy (all rows)
  category        98.5%
  priority        97.0%
  amount_paise    99.5%
  txn_date        98.0%
  txn_id         100.0%
  channel        100.0%
  language  

In [41]:
t = time.time()
test_preds = T.generate(model, tok, test, batch_size=1)
T.write_jsonl(test_preds, f"{SUB}/predictions_test.jsonl")
shutil.copy(f"{SUB}/predictions_test.jsonl", f"{WORK}/predictions_test.jsonl")
def _valid(o):
    try: return isinstance(json.loads(o), dict)
    except Exception: return False
print(f"test generation {time.time()-t:.0f}s; rows {len(test_preds)}; valid JSON {sum(_valid(p['output']) for p in test_preds)}/{len(test_preds)}")

  generated 1/400 (4s)
  generated 21/400 (84s)
  generated 41/400 (166s)
  generated 61/400 (249s)
  generated 81/400 (335s)
  generated 101/400 (419s)
  generated 121/400 (510s)
  generated 141/400 (604s)
  generated 161/400 (700s)
  generated 181/400 (800s)
  generated 201/400 (896s)
  generated 221/400 (995s)
  generated 241/400 (1096s)
  generated 261/400 (1208s)
  generated 281/400 (1319s)
  generated 301/400 (1431s)
  generated 321/400 (1538s)
  generated 341/400 (1650s)
  generated 361/400 (1764s)
  generated 381/400 (1877s)
test generation 1985s; rows 400; valid JSON 400/400


In [42]:
# Decoding sensitivity: the hub generation_config of Qwen2.5-Instruct sets repetition_penalty=1.05.
# Our predictions use pure greedy (repetition_penalty=1.0). Measure whether a grader calling
# model.generate(do_sample=False, max_new_tokens=200) with the hub defaults would get the same text.
from transformers import GenerationConfig
hub = GenerationConfig.from_pretrained(MODEL_ID); hub.do_sample = False; hub.max_new_tokens = 200
for k in ("temperature", "top_p", "top_k"): setattr(hub, k, None)
hub.eos_token_id, hub.pad_token_id = T.end_ids(tok), tok.pad_token_id   # isolate the penalty as the only difference
sub = dev[:50]
alt = {p["id"]: p["output"] for p in T.generate(model, tok, sub, batch_size=1, gen_cfg=hub)}
same = sum(alt[r["id"]] == {p["id"]: p["output"] for p in dev_preds}[r["id"]] for r in sub)
print(f"hub-default decoding (repetition_penalty={GenerationConfig.from_pretrained(MODEL_ID).repetition_penalty}) "
      f"identical on {same}/{len(sub)} dev tickets")

  generated 1/50 (4s)
  generated 21/50 (94s)
  generated 41/50 (206s)
hub-default decoding (repetition_penalty=1.1) identical on 49/50 dev tickets


In [43]:
# Reproducibility check (what the graders do): fresh base + adapter from disk, 25 random test rows, must match exactly
del model; torch.cuda.empty_cache()
m2 = T.load_for_inference(MODEL_ID, f"{SUB}/adapter", cfg["load_4bit"])
sample = random.Random(0).sample(test, 25)
again = {p["id"]: p["output"] for p in T.generate(m2, tok, sample, batch_size=1)}
saved = {p["id"]: p["output"] for p in test_preds}
mism = [i for i in again if again[i] != saved[i]]
print(f"reproduced {25-len(mism)}/25 exactly", mism)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

  generated 1/25 (4s)
  generated 21/25 (97s)
reproduced 25/25 exactly []


## Part E - Error analysis (best model on dev)

In [44]:
gold = {r["id"]: json.loads(r["messages"][-1]["content"]) for r in dev}
text = {r["id"]: r["messages"][1]["content"] for r in dev}
KEYS = ["category", "priority", "amount_paise", "txn_date", "txn_id", "channel", "language", "needs_human"]
def parse(o):
    try: x = json.loads(o); return x if isinstance(x, dict) else None
    except Exception: return None
P = {p["id"]: parse(p["output"]) for p in dev_preds}
fails = []
for i, g in gold.items():
    p = P.get(i)
    bad = [k for k in KEYS if p is None or k not in p or type(p[k]) is not type(g[k]) or p[k] != g[k]]
    if bad: fails.append((i, bad))
print(f"exact {1-len(fails)/len(gold):.1%}; failing rows {len(fails)}")
print("wrong-field counts:", collections.Counter(k for _, b in fails for k in b).most_common())
# per gold category: rows, exact-match rate, and which fields fail
failmap = dict(fails)
percat = collections.defaultdict(lambda: {"n": 0, "exact": 0, "wrong fields": collections.Counter()})
for i, g in gold.items():
    d = percat[g["category"]]; d["n"] += 1; d["exact"] += i not in failmap; d["wrong fields"].update(failmap.get(i, []))
display(pd.DataFrame({c: {"n": d["n"], "exact": f"{d['exact']/d['n']:.0%}", "wrong fields": dict(d["wrong fields"])}
                      for c, d in percat.items()}).T.sort_values("n", ascending=False))
# category confusion
display(pd.crosstab(pd.Series([g["category"] for g in gold.values()], name="gold"),
                    pd.Series([(P[i] or {}).get("category", "INVALID") for i in gold], name="pred")))

exact 93.5%; failing rows 13
wrong-field counts: [('priority', 6), ('txn_date', 4), ('needs_human', 4), ('category', 3), ('amount_paise', 1)]


,n,exact,wrong fields
refund,55,91%,"{'category': 1, 'priority': 4, 'needs_human': 3}"
payment_failed,53,87%,"{'txn_date': 4, 'category': 2, 'priority': 2, ..."
kyc,24,100%,{}
account_access,19,100%,{}
offers,18,94%,{'amount_paise': 1}
other,16,100%,{}
fraud,15,100%,{}


pred,account_access,fraud,kyc,offers,other,payment_failed,product,refund
gold,,,,,,,,
account_access,19,0,0,0,0,0,0,0
fraud,0,15,0,0,0,0,0,0
kyc,0,0,24,0,0,0,0,0
offers,0,0,0,18,0,0,0,0
other,0,0,0,0,16,0,0,0
payment_failed,0,2,0,0,0,51,0,0
refund,0,0,0,0,0,0,1,54


In [45]:
# Slices of dev by hard feature (computed from the ticket text with the spec checker)
def feats(u):
    c = speccheck.check(u); own = c["own"]
    return {"quoted thread": "\n>" in u, "hinglish": c["hinglish"] >= 0.05, "relative date": any(k.startswith("rel") for _, k in c["dates"]),
            "year-less date": any(k in ("dMy", "Mdy") and not re.search(r"20\d\d", own) for _, k in c["dates"]),
            "distractor amount": bool(c["distractor_amounts"]), "legal threat": c["legal"], "repeat contact": c["repeat"],
            "lakh/k": bool(re.search(r"(?i)lakh|\dk\b", own)), "spaced/odd txn id": bool(re.search(r"(?i)n\W*p\W+\d", own))}
fs = {i: feats(text[i]) for i in gold}; failset = {i for i, _ in fails}
display(pd.DataFrame({k: {"n": sum(f[k] for f in fs.values()),
                          "exact": f"{np.mean([i not in failset for i in gold if fs[i][k]]):.0%}" if any(f[k] for f in fs.values()) else "-"}
                      for k in next(iter(fs.values()))}).T)

,n,exact
quoted thread,3,67%
hinglish,49,94%
relative date,28,93%
year-less date,28,82%
distractor amount,12,100%
legal threat,8,100%
repeat contact,26,96%
lakh/k,10,100%
spaced/odd txn id,54,91%


In [46]:
# Every failure, with the fields that differ (write the explanation for >=10 of these in the report)
for i, bad in fails:
    g, p = gold[i], P[i]
    print(f"### {i}  wrong: {bad}")
    print("   ", re.sub(r"\s+", " ", speccheck.split_ticket(text[i])[2])[:300])
    print("    gold:", {k: g[k] for k in bad})
    if p is None:
        print("    pred (not valid JSON):", next(d["output"] for d in dev_preds if d["id"] == i)[:200])
    else:
        print("    pred:", {k: p.get(k, "<missing>") for k in bad})

### dv-00005  wrong: ['txn_date']
    [IVR transcript] Hey, I tried to make a transfer but it failed. Dated Aug 1. Total 49680 rupees. Ref: NP9512428472 Kindly resolve this asap.
    gold: {'txn_date': '2026-08-01'}
    pred: {'txn_date': '2026-07-01'}
### dv-00010  wrong: ['category', 'priority', 'needs_human']
    Subject: Problem with NimbusPay URGENT!! I didn't see the money I added to my card. ID: NP0194178721 INR 2,570 was the value. Please look into it. Regards, Vikram Singh
    gold: {'category': 'payment_failed', 'priority': 'P3', 'needs_human': False}
    pred: {'category': 'fraud', 'priority': 'P1', 'needs_human': True}
### dv-00019  wrong: ['category']
    Dear NimbusPay support, My product was faulty. Amount: 259419 rupees I am on the iOS app. Mode: credit card. Kindly resolve this asap.
    gold: {'category': 'refund'}
    pred: {'category': 'product'}
### dv-00037  wrong: ['priority', 'needs_human']
    My transaction history is looking a little off. I have been charged 

In [47]:
# Package everything for the repo (adapter/, predictions_test.jsonl, cleaning_log.csv, experiments.md, logs)
shutil.copy(f"{FINAL_DIR}/train_log.json", f"{SUB}/final_train_log.json")
if os.path.exists(f"{WORK}/baselines/results.json"): shutil.copy(f"{WORK}/baselines/results.json", f"{SUB}/baselines.json")
for name in EXPERIMENTS:
    p = f"{WORK}/exp/{name}/result.json"
    if os.path.exists(p): os.makedirs(f"{SUB}/logs", exist_ok=True); shutil.copy(p, f"{SUB}/logs/{name}.json")
shutil.make_archive("/content/submission_bundle" if os.path.isdir("/content") else "submission_bundle", "zip", SUB)
print(sorted(os.listdir(SUB)))

['adapter', 'baselines.json', 'cleaning_log.csv', 'experiments.md', 'final_train_log.json', 'logs', 'predictions_dev.jsonl', 'predictions_test.jsonl']
